In [2]:
import os
import urllib.parse
import pandas as pd
from sqlalchemy import create_engine, text

SERVER = r"DANNYPC\SQLEXPRESS"
DATABASE = "MAG7_MarketDB"
DRIVER = "ODBC Driver 17 for SQL Server"
PRICE_DATA_DIR = r"C:\Users\Admin\Desktop\Project\data\price_data"

params = urllib.parse.quote_plus(
    f"DRIVER={{{DRIVER}}};"
    f"SERVER={SERVER};"
    f"DATABASE={DATABASE};"
    "Trusted_Connection=yes;"
)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}",
    fast_executemany=True
)

print("Connected to:", pd.read_sql("SELECT DB_NAME() AS db_name", engine).iloc[0, 0])

for filename in os.listdir(PRICE_DATA_DIR):
    if not filename.endswith(".csv"):
        continue

    ticker = filename.split("_")[0]
    file_path = os.path.join(PRICE_DATA_DIR, filename)

    print(f"Loading {ticker}...")

    df = pd.read_csv(file_path)
    df["ticker"] = ticker
    df["Date"] = pd.to_datetime(df["Date"]).dt.date

    out = df.rename(columns={
        "Date": "date",
        "Open": "open",
        "High": "high",
        "Low": "low",
        "Close": "close",
        "Adj Close": "adj_close",
        "Volume": "volume"
    })

    out = out[[
        "ticker", "date", "open", "high", "low",
        "close", "adj_close", "volume"
    ]].dropna()

    with engine.begin() as conn:
        conn.execute(
            text("DELETE FROM dbo.prices_daily WHERE ticker = :ticker"),
            {"ticker": ticker}
        )

    out.to_sql(
        "prices_daily",
        con=engine,
        schema="dbo",
        if_exists="append",
        index=False,
        chunksize=500
    )

    print(f"{ticker}: {len(out)} rows loaded.")

print("Done.")


Connected to: MAG7_MarketDB
Loading AAPL...
AAPL: 11423 rows loaded.
Loading AMZN...
AMZN: 7271 rows loaded.
Loading GOOGL...
GOOGL: 5445 rows loaded.
Loading META...
META: 3493 rows loaded.
Loading MSFT...
MSFT: 10097 rows loaded.
Loading NVDA...
NVDA: 6846 rows loaded.
Loading TSLA...
TSLA: 3970 rows loaded.
Done.
